# 01 — Exploratory Data Analysis
**Case Study 1 — AI-Driven Energy Intelligence for Industrial Buildings**

Goals (Week 1-2): understand the dataset structure, load patterns, seasonality,
and the industrial scenario before any modelling.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px

from src.data.ingestion import load_dataset
from src.data.cleaning import clean_dataset

# Default: synthetic data (offline). Switch to source="bdg2" for real BDG2 data.
ds = load_dataset(source="synthetic", n_buildings=8)
print(f"{ds.meters.shape[0]:,} readings | {ds.metadata.shape[0]} buildings")
ds.metadata

## Consumption overview

In [ ]:
clean, _ = clean_dataset(ds.meters)
daily = (clean.set_index("timestamp").groupby("building_id")["meter_reading"]
         .resample("D").sum().reset_index())
px.line(daily, x="timestamp", y="meter_reading", color="building_id",
        title="Daily consumption per building (kWh)")

## Load profile: hour of day × weekday
Industrial buildings show shift patterns — high weekday day-load, low nights/weekends.

In [ ]:
b = clean["building_id"].iloc[0]
one = clean[clean["building_id"] == b].copy()
one["hour"] = one["timestamp"].dt.hour
one["weekday"] = one["timestamp"].dt.day_name()
pivot = one.pivot_table(index="weekday", columns="hour", values="meter_reading", aggfunc="mean")
pivot = pivot.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
px.imshow(pivot, aspect="auto", color_continuous_scale="YlOrRd", title=f"Mean load profile — {b}")

## Weather relationship

In [ ]:
merged = one.merge(ds.weather, on="timestamp")
sample = merged.sample(min(3000, len(merged)), random_state=1)
px.scatter(sample, x="air_temperature", y="meter_reading", opacity=0.3,
           title=f"Load vs outdoor temperature — {b} (V-shape = electric heating + cooling)")

## Findings
- Clear daily/weekly seasonality driven by shift operation.
- Temperature has a V-shaped effect (heating + cooling) → useful forecast feature.
- Raw data contains visible artefacts → quantified in `02_data_quality.ipynb`.